# CSoT'26 — ML in Astronomy — Week 1 · Part 2: Data Pipeline (Solution)

Reference implementation. **Only open after attempting [`week1_data_starter.ipynb`](week1_data_starter.ipynb).**

Structure mirrors the starter, with code filled in and short commentary on the *why*. Companion reading: [`08-data-pipelines.md`](../08-data-pipelines.md).

## Step 0 — Imports

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## Step 1 — Get the dataset into Colab

Using the Kaggle API. You need a free Kaggle account and an API token (`kaggle.json` from Account → Create New API Token).

If you've already downloaded/unzipped in a previous session (e.g. to Drive), just set `DATA_ROOT` and skip the download.

In [ ]:
# --- Download via Kaggle API (run once) ---
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
# !unzip -q galaxy-zoo-2-images.zip -d galaxy_data

# Point this at the folder that directly contains the class subfolders.
# Adjust to match how the archive unzipped on your machine.
DATA_ROOT = "galaxy_data"
print("DATA_ROOT =", DATA_ROOT)

## Step 2 — Inspect the folder structure

Always look at the raw layout before trusting `ImageFolder`. We expect `DATA_ROOT/<class>/<image>.jpg`.

In [ ]:
for entry in sorted(os.listdir(DATA_ROOT)):
    path = os.path.join(DATA_ROOT, entry)
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"{entry:20s} {n:6d} images")

## Step 3 — Transforms pipeline

Order matters: `Resize` (known size) → `ToTensor` (PIL→tensor, 0-255→0-1) → `Normalize` (per-channel standardisation). With mean=std=0.5 the 0–1 range maps to roughly [-1, 1].

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

## Step 4 — ImageFolder

`ImageFolder` reads labels from the folder names and assigns indices alphabetically. Always inspect `class_to_idx` so you can map predictions back to names later.

In [ ]:
dataset = ImageFolder(root=DATA_ROOT, transform=transform)
print("num images   :", len(dataset))
print("classes      :", dataset.classes)
print("class_to_idx :", dataset.class_to_idx)

## Step 5 — A single sample

Each item is a `(image_tensor, label_int)` tuple. The transform was applied lazily inside `__getitem__`.

In [ ]:
image, label = dataset[0]
print("shape :", image.shape)   # torch.Size([3, 64, 64])
print("dtype :", image.dtype)   # torch.float32
print("label :", label, "->", dataset.classes[label])

## Step 6 — DataLoader and one batch

The `DataLoader` shuffles and batches. `next(iter(loader))` pulls a single batch for inspection — the leading dimension is the batch size.

In [ ]:
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

images, labels = next(iter(loader))
print("images:", images.shape)   # (32, 3, 64, 64) = (B, C, H, W)
print("labels:", labels.shape)   # (32,)

## Step 7 — Plot a batch

The single most valuable sanity check before any training. Two recurring gotchas are handled here:
1. Undo `Normalize` (`x*0.5 + 0.5`) so colours look right.
2. `permute(1, 2, 0)` because matplotlib wants `(H, W, C)`, not PyTorch's `(C, H, W)`.

In [ ]:
images_show = images * 0.5 + 0.5          # undo Normalize([0.5],[0.5])
grid = torchvision.utils.make_grid(images_show[:16], nrow=4)

plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("A batch of galaxies")
plt.show()

print("Labels:", [dataset.classes[i] for i in labels[:16].tolist()])

## Stretch Goal 1 — Reproducible train/val split

Always seed the split so experiments are reproducible.

In [ ]:
n_val = int(0.15 * len(dataset))
n_train = len(dataset) - n_val
train_subset, val_subset = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)
print(f"train: {len(train_subset)}  val: {len(val_subset)}")

## Stretch Goal 2 — Real per-channel mean/std

Compute the dataset's actual statistics instead of assuming 0.5. Uses a fresh dataset with only `Resize` + `ToTensor` (no normalisation), so we measure the true 0–1 distribution.

In [ ]:
stat_ds = ImageFolder(
    root=DATA_ROOT,
    transform=transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()]),
)
stat_loader = DataLoader(stat_ds, batch_size=64, shuffle=False, num_workers=2)

n_pixels = 0
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
for imgs, _ in stat_loader:
    # imgs: (B, 3, H, W); sum over batch + spatial dims
    channel_sum += imgs.sum(dim=[0, 2, 3])
    channel_sq_sum += (imgs ** 2).sum(dim=[0, 2, 3])
    n_pixels += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]

mean = channel_sum / n_pixels
std = (channel_sq_sum / n_pixels - mean ** 2).sqrt()
print("per-channel mean:", mean.tolist())
print("per-channel std :", std.tolist())
print("(Compare to the 0.5 / 0.5 we assumed.)")

## Stretch Goal 4 — Augmentation preview

Galaxies have no canonical orientation, so flips and rotations are physically reasonable augmentations — unlike, say, photos of text. Apply a train-only transform to one image several times and watch it change.

In [ ]:
aug = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(180),
    transforms.ToTensor(),
])

# Grab the raw PIL image for the first sample (index into the underlying samples list).
img_path, _ = dataset.samples[0]
from PIL import Image
pil_img = Image.open(img_path).convert("RGB")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax in axes:
    ax.imshow(aug(pil_img).permute(1, 2, 0).numpy())
    ax.axis("off")
fig.suptitle("Same galaxy, five random augmentations")
plt.show()

## Reflection (example answers)

Yours will differ — these are illustrative.

1. **Most confusing part.** Getting `ImageFolder` to find images — the archive unzipped one directory deeper than expected, so `DATA_ROOT` needed adjusting. Printing `os.listdir` revealed it instantly.
2. **Recognising a spiral.** A CNN needs to detect localised, multi-scale features: a bright central bulge, two or more curved arms with a measurable pitch, and possibly dust lanes silhouetted against the disk.
3. **Hardest pair.** Likely **lenticulars (S0) vs ellipticals** — a face-on S0 looks almost identical to an elliptical without visible arms. We'll confirm this against the confusion matrix later.
4. **Why the pipeline first.** A subtle data bug (wrong normalisation, leaked val set, mislabelled folders) silently destroys model quality without raising an error. Getting data right before adding a model means there's only one moving part to debug at a time.

---

That completes Week 1. Next up: **Week 2 — Baselines & Fully-Connected Networks** (theory coming soon).